In [2]:
import json
from pathlib import Path

DATA_PATH = Path(
    r"G:\SEVA_Ai\SEVA_Ai\notebooks\data\analysis\all_schemes.json"
)

with open(DATA_PATH, "r", encoding="utf-8") as f:
    schemes = json.load(f)

print("Total schemes:", len(schemes))
print("First scheme fields:", list(schemes[0].keys()))

Total schemes: 4428
First scheme fields: ['scheme_id', 'internal_id', 'slug', 'name', 'short_title', 'level', 'ministry', 'nodal_department', 'department', 'states', 'categories', 'tags', 'target_beneficiaries', 'status', 'open_date', 'close_date', 'brief_description', 'details', 'benefits', 'exclusions', 'eligibility', 'application_process', 'documents_required', 'faqs', 'apply_url', 'apply_links', 'official_links', 'myscheme_url', 'source', 'fetched_on']


In [3]:
from collections import Counter

fields_to_inspect = [
    "brief_description",
    "details",
    "benefits",
    "target_beneficiaries",
    "eligibility",
    "exclusions",
    "documents_required",
    "application_process",
    "faqs",
    "categories",
    "states",
    "tags"
]

for field in fields_to_inspect:
    types = Counter(
        type(scheme.get(field)).__name__
        for scheme in schemes
    )

    print(f"\n{field}: {dict(types)}")


brief_description: {'str': 4428}

details: {'str': 4428}

benefits: {'str': 4428}

target_beneficiaries: {'list': 4368, 'NoneType': 60}

eligibility: {'str': 4428}

exclusions: {'str': 4428}

documents_required: {'list': 4428}

application_process: {'list': 4428}

faqs: {'list': 4428}

categories: {'list': 4428}

states: {'list': 4428}

tags: {'list': 4428}



## Step 3: Inspect Nested Data Structures

**Purpose:**
Understand the internal structure of list-based fields, especially documents, application steps, FAQs, beneficiaries, categories, and states.

This helps us design reliable extraction and semantic grouping logic for child document generation.


In [4]:

# Inspect representative list-based fields from the first scheme

list_fields = [
    "target_beneficiaries",
    "documents_required",
    "application_process",
    "faqs",
    "categories",
    "states",
    "tags"
]

for field in list_fields:
    print(f"\n{'=' * 70}")
    print(f"FIELD: {field}")
    print(f"{'=' * 70}")

    value = schemes[0].get(field)

    print("Number of items:", len(value) if isinstance(value, list) else "N/A")
    print("Sample value:")
    print(json.dumps(value, indent=2, ensure_ascii=False)[:3000])



FIELD: target_beneficiaries
Number of items: 2
Sample value:
[
  {
    "value": "individual",
    "label": "Individual"
  },
  {
    "value": "business-entity",
    "label": "Business Entity"
  }
]

FIELD: documents_required
Number of items: 17
Sample value:
[
  "Proof of Identity: Voter’s ID Card / Passport / Driving License / PAN Card / Signature identification from present bankers of proprietor, partner of director ( if a company)",
  "Proof of Residence: Recent telephone bills, electricity bill, property tax receipt /Passport / voter’s ID Card of Proprietor, partner of Director (if a company)",
  "Proof of Business Address",
  "Proof that the applicant is not a defaulter in any Bank / Financial Institution",
  "Memorandum and articles of association of the Company / Partnership Deed of partners etc.",
  "Assets and liabilities statement of promoters and guarantors along with latest income tax returns.",
  "Rent Agreement (if business premises on rent) and clearance from pollution 


## Step 4: Inspect List-Based Field Contents

**Purpose:**
Inspect the actual items and nested structures in documents_required, application_process, faqs, and target_beneficiaries.

We need this information to define semantic grouping and preserve FAQ question-answer pairs during child generation.


In [5]:

# Inspect the first scheme's list-based fields

inspect_fields = [
    "target_beneficiaries",
    "documents_required",
    "application_process",
    "faqs"
]

for field in inspect_fields:
    value = schemes[0].get(field)

    print(f"\n{'=' * 70}")
    print(f"FIELD: {field}")
    print(f"ITEM COUNT: {len(value) if isinstance(value, list) else 'N/A'}")
    print(f"{'=' * 70}")

    if isinstance(value, list):
        for i, item in enumerate(value[:5], start=1):
            print(f"\nItem {i}:")
            print(json.dumps(item, indent=2, ensure_ascii=False))
    else:
        print(value)



FIELD: target_beneficiaries
ITEM COUNT: 2

Item 1:
{
  "value": "individual",
  "label": "Individual"
}

Item 2:
{
  "value": "business-entity",
  "label": "Business Entity"
}

FIELD: documents_required
ITEM COUNT: 17

Item 1:
"Proof of Identity: Voter’s ID Card / Passport / Driving License / PAN Card / Signature identification from present bankers of proprietor, partner of director ( if a company)"

Item 2:
"Proof of Residence: Recent telephone bills, electricity bill, property tax receipt /Passport / voter’s ID Card of Proprietor, partner of Director (if a company)"

Item 3:
"Proof of Business Address"

Item 4:
"Proof that the applicant is not a defaulter in any Bank / Financial Institution"

Item 5:
"Memorandum and articles of association of the Company / Partnership Deed of partners etc."

FIELD: application_process
ITEM COUNT: 1

Item 1:
{
  "mode": "Online",
  "steps": "Either approach your nearest bank branch to apply (locate your nearest bank here - [https://www.rbi.org.in/Scr


## Step 5: Validate List Structures Across All Schemes

**Purpose:**
Check whether list-based fields have consistent item types and dictionary keys across the complete dataset.

This prevents extraction errors when generating discovery and detail children.


In [7]:

from collections import Counter

list_fields = [
    "target_beneficiaries",
    "documents_required",
    "application_process",
    "faqs"
]

for field in list_fields:
    item_types = Counter()
    dictionary_keys = Counter()
    empty_lists = 0

    for scheme in schemes:
        items = scheme.get(field)

        if not isinstance(items, list):
            continue

        if not items:
            empty_lists += 1

        for item in items:
            item_types[type(item).__name__] += 1

            if isinstance(item, dict):
                dictionary_keys[tuple(sorted(item.keys()))] += 1

    print(f"\n{'=' * 60}")
    print(f"FIELD: {field}")
    print("Item types:", dict(item_types))
    print("Dictionary structures:", dict(dictionary_keys))
    print("Empty lists:", empty_lists)



FIELD: target_beneficiaries
Item types: {'dict': 5298}
Dictionary structures: {('label', 'value'): 5298}
Empty lists: 0

FIELD: documents_required
Item types: {'str': 36493}
Dictionary structures: {}
Empty lists: 14

FIELD: application_process
Item types: {'dict': 4863}
Dictionary structures: {('mode', 'steps'): 4863}
Empty lists: 4

FIELD: faqs
Item types: {'dict': 48696}
Dictionary structures: {('answer', 'question'): 48696}
Empty lists: 12



## Step 6: Initialize Child Dataset Output Directories

**Purpose:** Create separate storage locations for discovery and detail child datasets without modifying the original source data.


In [8]:

OUTPUT_DIR = Path("G:/SEVA_Ai/SEVA_Ai/output/SEVA_CHILDREN")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DISCOVERY_PATH = OUTPUT_DIR / "discovery_children.jsonl"
DETAIL_PATH = OUTPUT_DIR / "detail_children.jsonl"

print("Output directory:", OUTPUT_DIR.resolve())
print("Discovery dataset:", DISCOVERY_PATH)
print("Detail dataset:", DETAIL_PATH)


Output directory: G:\SEVA_Ai\SEVA_Ai\output\SEVA_CHILDREN
Discovery dataset: G:\SEVA_Ai\SEVA_Ai\output\SEVA_CHILDREN\discovery_children.jsonl
Detail dataset: G:\SEVA_Ai\SEVA_Ai\output\SEVA_CHILDREN\detail_children.jsonl



## Step 7: Generate Discovery and Detail Child Documents

**Purpose:**
Transform the 4,428 original scheme records into structured child documents for two retrieval indexes.

Discovery children support citizen-problem-based scheme discovery. Detail children support follow-up questions about documents, application procedures, and FAQs.

Each child retains its scheme_id, parent_id, deterministic child_id, child_type, source_field, and useful filtering metadata.

FAQ question-answer pairs and application modes with their steps must remain together.


In [9]:

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd

print("Document:", Document)
print("Text splitter:", RecursiveCharacterTextSplitter)
print("Pandas version:", pd.__version__)


Document: <class 'langchain_core.documents.base.Document'>
Text splitter: <class 'langchain_text_splitters.character.RecursiveCharacterTextSplitter'>
Pandas version: 3.1.1



## Step 10: Define Structure-Aware Eligibility Chunking

**Purpose:**
Preserve meaningful eligibility conditions and their context during child document creation.

Rules:
1. Preserve source headings and labels.
2. Keep related conditions together.
3. Do not separate a condition from its explanation.
4. Preserve income limits, age limits, beneficiary categories, and other numerical constraints.
5. Use RecursiveCharacterTextSplitter only when a logical group exceeds the configured chunk size.
6. Retain scheme_id, parent_id, child_id, and source metadata.
7. Never invent headings or eligibility conditions.


In [10]:

from langchain_core.documents import Document

discovery_fields = {
    "overview": "brief_description",
    "benefits": "benefits",
    "beneficiaries": "target_beneficiaries",
    "eligibility": "eligibility",
    "exclusions": "exclusions",
}

discovery_documents = []

for scheme in schemes:
    scheme_id = str(scheme["scheme_id"])

    for child_type, source_field in discovery_fields.items():
        value = scheme.get(source_field)

        # Convert beneficiary dictionaries into readable labels
        if child_type == "beneficiaries":
            content = "; ".join(
                item["label"]
                for item in (value or [])
                if isinstance(item, dict) and item.get("label")
            )
        else:
            content = value.strip() if isinstance(value, str) else ""

        if not content:
            continue

        discovery_documents.append(
            Document(
                page_content=(
                    f"Scheme: {scheme.get('name', '')}\n"
                    f"Section: {child_type}\n"
                    f"{content}"
                ),
                metadata={
                    "scheme_id": scheme_id,
                    "parent_id": scheme_id,
                    "child_type": child_type,
                    "source_field": source_field,
                    "categories": scheme.get("categories") or [],
                    "states": scheme.get("states") or [],
                    "level": scheme.get("level", ""),
                    "language": "en",
                },
            )
        )

print("Discovery documents:", len(discovery_documents))
print("\nSample document:")
print(discovery_documents[0].page_content[:1000])
print("\nMetadata:")
print(discovery_documents[0].metadata)


Discovery documents: 18393

Sample document:
Scheme: Stand-Up India
Section: overview
A scheme by Ministry of Finance for financing SC/ST and Women Entrepreneurs by facilitating bank loans for setting up a greenfield project Enterprise in manufacturing, services, trading sector and activities allied to agriculture.

Metadata:
{'scheme_id': 'LBezF5gB5uF80_MGGY81', 'parent_id': 'LBezF5gB5uF80_MGGY81', 'child_type': 'overview', 'source_field': 'brief_description', 'categories': ['Business & Entrepreneurship', 'Banking,Financial Services and Insurance', 'Social welfare & Empowerment'], 'states': ['All'], 'level': 'Central', 'language': 'en'}



## Step 12: Create Structure-Aware Detail Documents

**Purpose:**
Create LangChain Document objects for documents_required, application_process, and FAQs.

Processing rules:
- Group document descriptions as readable lists.
- Preserve application modes and their associated steps.
- Keep FAQ question-answer pairs atomic.
- Preserve scheme-level metadata.
- Skip empty records.

**Output:** detail_documents (List[Document])


In [11]:

detail_documents = []

for scheme in schemes:
    scheme_id = str(scheme["scheme_id"])

    base_metadata = {
        "scheme_id": scheme_id,
        "parent_id": scheme_id,
        "categories": scheme.get("categories") or [],
        "states": scheme.get("states") or [],
        "level": scheme.get("level", ""),
        "language": "en",
    }

    scheme_name = scheme.get("name", "")

    # 1. Documents required
    documents = [
        item.strip()
        for item in (scheme.get("documents_required") or [])
        if isinstance(item, str) and item.strip()
    ]

    if documents:
        detail_documents.append(
            Document(
                page_content=(
                    f"Scheme: {scheme_name}\n"
                    f"Section: Documents Required\n"
                    + "\n".join(f"- {item}" for item in documents)
                ),
                metadata={
                    **base_metadata,
                    "child_type": "documents_required",
                    "source_field": "documents_required",
                },
            )
        )

    # 2. Application process: preserve mode and steps
    for index, process in enumerate(
        scheme.get("application_process") or [], start=1
    ):
        if not isinstance(process, dict):
            continue

        mode = process.get("mode", "")
        steps = process.get("steps")

        if isinstance(steps, list):
            steps_text = "\n".join(
                f"- {step}" for step in steps if step
            )
        elif isinstance(steps, str):
            steps_text = steps.strip()
        else:
            steps_text = str(steps) if steps is not None else ""

        content = (
            f"Scheme: {scheme_name}\n"
            f"Section: Application Process\n"
            f"Mode: {mode}\n"
            f"Steps:\n{steps_text}"
        ).strip()

        if mode or steps_text:
            detail_documents.append(
                Document(
                    page_content=content,
                    metadata={
                        **base_metadata,
                        "child_type": "application_process",
                        "source_field": "application_process",
                        "process_index": index,
                        "application_mode": str(mode),
                    },
                )
            )

    # 3. FAQs: preserve question-answer pairs
    for index, faq in enumerate(scheme.get("faqs") or [], start=1):
        if not isinstance(faq, dict):
            continue

        question = str(faq.get("question") or "").strip()
        answer = str(faq.get("answer") or "").strip()

        if not question and not answer:
            continue

        detail_documents.append(
            Document(
                page_content=(
                    f"Scheme: {scheme_name}\n"
                    f"Question: {question}\n"
                    f"Answer: {answer}"
                ),
                metadata={
                    **base_metadata,
                    "child_type": "faq",
                    "source_field": "faqs",
                    "faq_index": index,
                },
            )
        )

print("Detail documents:", len(detail_documents))
print("\nDocument type distribution:")
print(
    pd.Series(
        [doc.metadata["child_type"] for doc in detail_documents]
    ).value_counts()
)

print("\nSample detail document:")
print(detail_documents[0].page_content[:1500])


Detail documents: 57973

Document type distribution:
faq                    48696
application_process     4863
documents_required      4414
Name: count, dtype: int64

Sample detail document:
Scheme: Stand-Up India
Section: Documents Required
- Proof of Identity: Voter’s ID Card / Passport / Driving License / PAN Card / Signature identification from present bankers of proprietor, partner of director ( if a company)
- Proof of Residence: Recent telephone bills, electricity bill, property tax receipt /Passport / voter’s ID Card of Proprietor, partner of Director (if a company)
- Proof of Business Address
- Proof that the applicant is not a defaulter in any Bank / Financial Institution
- Memorandum and articles of association of the Company / Partnership Deed of partners etc.
- Assets and liabilities statement of promoters and guarantors along with latest income tax returns.
- Rent Agreement (if business premises on rent) and clearance from pollution control board if applicable.
- SSI / MS


## Step 13: Analyze Document Length Distribution

**Purpose:**
Use Pandas to inspect document lengths by child type and identify which documents need recursive splitting.

This helps avoid unnecessarily splitting short documents and FAQ pairs.


In [12]:

all_documents = discovery_documents + detail_documents

length_df = pd.DataFrame([
    {
        "child_type": doc.metadata["child_type"],
        "char_length": len(doc.page_content)
    }
    for doc in all_documents
])

print("Document count:", len(length_df))
print("\nLength statistics by child type:")

print(
    length_df.groupby("child_type")["char_length"]
    .agg(["count", "mean", "median", "max"])
    .round(2)
)

print("\nDocuments exceeding 800 characters:")

print(
    length_df.assign(
        oversized=length_df["char_length"] > 800
    )
    .groupby("child_type")["oversized"]
    .agg(["sum", "mean"])
    .round(3)
)


Document count: 76366

Length statistics by child type:
                     count     mean  median    max
child_type                                        
application_process   4863  1411.53  1163.0  11560
beneficiaries         4368   103.90    94.0    385
benefits              4423   673.98   424.0  28390
documents_required    4414   587.14   402.5   8150
eligibility           4424   735.83   555.5   9395
exclusions             750   470.59   322.5   3938
faq                  48696   297.34   273.0   9078
overview              4428   326.38   334.0    665

Documents exceeding 800 characters:
                      sum   mean
child_type                      
application_process  3645  0.750
beneficiaries           0  0.000
benefits             1105  0.250
documents_required    892  0.202
eligibility          1294  0.292
exclusions            100  0.133
faq                   253  0.005
overview                0  0.000



## Step 14: Create Parent Documents and Parent Storage

**Purpose:**
Create complete semantic parent records from the existing discovery and detail LangChain Documents.

Architecture:
- One parent per discovery section.
- One parent per complete documents-required section.
- One parent per application mode.
- One parent per FAQ question-answer pair.

Each parent receives a deterministic parent_id. The corresponding LangChain Document receives the same parent_id in its metadata.

The complete parent content is preserved independently of subsequent chunking.

Output: parent_documents.jsonl


In [13]:

import json

parent_records = []

def create_parent_record(doc):
    metadata = doc.metadata
    scheme_id = metadata["scheme_id"]
    child_type = metadata["child_type"]

    # Generate a stable section-level parent ID
    if child_type in ["application_process", "faq"]:
        index = metadata.get(
            "process_index" if child_type == "application_process"
            else "faq_index"
        )
        parent_id = f"{scheme_id}_{child_type.upper()}_{index:03d}"
    else:
        parent_id = f"{scheme_id}_{child_type.upper()}_001"

    # Link original semantic document to its parent
    metadata["parent_id"] = parent_id
    metadata["child_id"] = parent_id

    return {
        "parent_id": parent_id,
        "scheme_id": scheme_id,
        "child_type": child_type,
        "source_field": metadata["source_field"],
        "full_text": doc.page_content,
        "metadata": metadata.copy(),
    }


# Create parents from both datasets
for doc in discovery_documents + detail_documents:
    parent_records.append(create_parent_record(doc))


# Validate parent IDs
parent_ids = [record["parent_id"] for record in parent_records]

print("Total parent records:", len(parent_records))
print("Unique parent IDs:", len(set(parent_ids)))
print("Duplicate parent IDs:", len(parent_ids) - len(set(parent_ids)))

# Save complete parent records
PARENT_PATH = OUTPUT_DIR / "parent_documents.jsonl"

with open(PARENT_PATH, "w", encoding="utf-8") as f:
    for record in parent_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("\nParent storage saved:", PARENT_PATH.resolve())


Total parent records: 76366
Unique parent IDs: 76366
Duplicate parent IDs: 0

Parent storage saved: G:\SEVA_Ai\SEVA_Ai\output\SEVA_CHILDREN\parent_documents.jsonl



## Step 15: Generate Retrieval Chunks Using LangChain

**Purpose:**
Split oversized parent documents using RecursiveCharacterTextSplitter while preserving complete parent relationships.

Rules:
- Documents under 800 characters remain intact.
- Long documents are recursively split.
- Each chunk receives a deterministic chunk_id.
- Each chunk retains its parent_id and scheme_id.
- FAQ questions are preserved in every corresponding answer chunk.
- Complete parent content remains unchanged in parent storage.

Output:
- discovery_chunks
- detail_chunks


In [15]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    add_start_index=True
)

print("Text splitter initialized successfully.")

def generate_chunks(documents):
    chunks = []

    for doc in documents:
        parent_id = doc.metadata["parent_id"]

        split_docs = (
            [doc]
            if len(doc.page_content) <= 800
            else text_splitter.split_documents([doc])
        )

        for index, chunk in enumerate(split_docs, start=1):
            chunk.metadata["parent_id"] = parent_id
            chunk.metadata["chunk_id"] = f"{parent_id}_CHUNK_{index:03d}"
            chunk.metadata["chunk_index"] = index

            chunks.append(chunk)

    return chunks


discovery_chunks = generate_chunks(discovery_documents)
detail_chunks = generate_chunks(detail_documents)

print("Discovery chunks:", len(discovery_chunks))
print("Detail chunks:", len(detail_chunks))
print("Total retrieval chunks:", len(discovery_chunks) + len(detail_chunks))


Text splitter initialized successfully.
Discovery chunks: 23438
Detail chunks: 67945
Total retrieval chunks: 91383



## Step 17: Validate Retrieval Chunks

**Purpose:**
Validate chunk lengths, unique chunk IDs, parent references, and FAQ chunk relationships before exporting the datasets.


In [16]:

all_chunks = discovery_chunks + detail_chunks

chunk_df = pd.DataFrame([
    {
        "chunk_id": doc.metadata["chunk_id"],
        "parent_id": doc.metadata["parent_id"],
        "scheme_id": doc.metadata["scheme_id"],
        "child_type": doc.metadata["child_type"],
        "length": len(doc.page_content),
    }
    for doc in all_chunks
])

parent_ids = {record["parent_id"] for record in parent_records}

print("Total chunks:", len(chunk_df))
print("Unique chunk IDs:", chunk_df["chunk_id"].nunique())
print("Duplicate chunk IDs:", chunk_df["chunk_id"].duplicated().sum())

print("Chunks exceeding 800 characters:", (chunk_df["length"] > 800).sum())

print("Missing parent references:",
      (~chunk_df["parent_id"].isin(parent_ids)).sum())

print("\nChunks per child type:")
print(chunk_df.groupby("child_type").agg(
    chunks=("chunk_id", "count"),
    average_length=("length", "mean"),
    maximum_length=("length", "max")
).round(2))

print("\nFAQ parents split into multiple chunks:")
print(
    chunk_df[chunk_df["child_type"] == "faq"]
    .groupby("parent_id").size().gt(1).sum()
)


Total chunks: 91383
Unique chunk IDs: 91383
Duplicate chunk IDs: 0
Chunks exceeding 800 characters: 0
Missing parent references: 0

Chunks per child type:
                     chunks  average_length  maximum_length
child_type                                                 
application_process   13127          531.46             800
beneficiaries          4368          103.90             385
benefits               6879          441.06             800
documents_required     5795          456.73             800
eligibility            6835          480.71             800
exclusions              928          384.76             799
faq                   49023          295.54             799
overview               4428          326.38             665

FAQ parents split into multiple chunks:
253



## Step 18: Export Discovery and Detail Retrieval Datasets

**Purpose:**
Export the validated LangChain Document chunks into separate JSONL files.

Each record contains:
- chunk_id
- parent_id
- scheme_id
- child_type
- text
- metadata

Complete parent sections remain stored separately in parent_documents.jsonl.

Output:
- discovery_children.jsonl
- detail_children.jsonl


In [17]:

def save_documents_jsonl(documents, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        for doc in documents:
            record = {
                "chunk_id": doc.metadata["chunk_id"],
                "parent_id": doc.metadata["parent_id"],
                "scheme_id": doc.metadata["scheme_id"],
                "child_type": doc.metadata["child_type"],
                "text": doc.page_content,
                "metadata": doc.metadata,
            }

            f.write(json.dumps(record, ensure_ascii=False) + "\n")


save_documents_jsonl(discovery_chunks, DISCOVERY_PATH)
save_documents_jsonl(detail_chunks, DETAIL_PATH)

print("Discovery dataset saved:", DISCOVERY_PATH.resolve())
print("Discovery chunks:", len(discovery_chunks))

print("\nDetail dataset saved:", DETAIL_PATH.resolve())
print("Detail chunks:", len(detail_chunks))


Discovery dataset saved: G:\SEVA_Ai\SEVA_Ai\output\SEVA_CHILDREN\discovery_children.jsonl
Discovery chunks: 23438

Detail dataset saved: G:\SEVA_Ai\SEVA_Ai\output\SEVA_CHILDREN\detail_children.jsonl
Detail chunks: 67945



## Step 19: Validate Exported JSONL Datasets

**Purpose:**
Reload all three JSONL files and validate record counts, unique IDs, and parent-child relationships.

This confirms that the exported datasets are ready for embedding and Chroma indexing.


In [18]:

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


saved_parents = load_jsonl(PARENT_PATH)
saved_discovery = load_jsonl(DISCOVERY_PATH)
saved_detail = load_jsonl(DETAIL_PATH)

parent_id_set = {p["parent_id"] for p in saved_parents}

all_saved_chunks = saved_discovery + saved_detail

print("Parent records:", len(saved_parents))
print("Discovery chunks:", len(saved_discovery))
print("Detail chunks:", len(saved_detail))

print("\nUnique parent IDs:",
      len(parent_id_set))

print("Unique chunk IDs:",
      len({c["chunk_id"] for c in all_saved_chunks}))

print("Missing parent references:",
      sum(c["parent_id"] not in parent_id_set
          for c in all_saved_chunks))

print("Invalid text records:",
      sum(not isinstance(c.get("text"), str) or not c["text"].strip()
          for c in all_saved_chunks))


Parent records: 76366
Discovery chunks: 23438
Detail chunks: 67945

Unique parent IDs: 76366
Unique chunk IDs: 91383
Missing parent references: 0
Invalid text records: 0



## Step 20: Initialize HuggingFace Embedding Model

**Purpose:**
Initialize the sentence-transformers embedding model using LangChain's HuggingFaceEmbeddings integration.

The same embedding configuration will be used for discovery and detail Chroma collections.

We will verify the embedding dimension before indexing.


In [19]:

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 16
    }
)

test_embedding = embedding_model.embed_query(
    "Government financial assistance for farmers"
)

print("Model:", "sentence-transformers/all-MiniLM-L6-v2")
print("Embedding dimension:", len(test_embedding))
print("Embedding initialized successfully.")


g:\SEVA_Ai\SEVA_Ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 716.42it/s]


Model: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384
Embedding initialized successfully.



## Step 21: Initialize Chroma Vector Stores

**Purpose:**
Create independent persistent Chroma collections for discovery and detail retrieval.

Use the same embedding model for both collections. Existing chunking experiment databases must remain untouched.


In [20]:

from langchain_chroma import Chroma

VECTORSTORE_DIR = OUTPUT_DIR / "vectorstores"
VECTORSTORE_DIR.mkdir(parents=True, exist_ok=True)

discovery_db = Chroma(
    collection_name="seva_discovery_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "discovery")
)

detail_db = Chroma(
    collection_name="seva_detail_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "detail")
)

print("Discovery collection:", discovery_db._collection.count())
print("Detail collection:", detail_db._collection.count())


Discovery collection: 0
Detail collection: 0



## Step 22: Batch Index Discovery and Detail Documents

**Purpose:**
Index both validated child datasets into their respective persistent Chroma collections.

Use batch processing to control RAM consumption and avoid processing all 91,383 documents simultaneously.

The operation uses LangChain Chroma and the configured Hugging Face embedding model.



## Step 22: Resumable Chroma Indexing with Checkpoints

**Purpose:**
Index discovery and detail documents using persistent, crash-safe batch processing.

Features:
- Batch size: 500 documents.
- Embedding batch size: 16.
- Separate checkpoint files.
- Checkpoint updated after every successful batch.
- Existing Chroma IDs checked before inserting.
- Missing documents are automatically indexed after interruption.
- Metadata lists and dictionaries are converted into Chroma-compatible values.

The Chroma collection is the source of truth for determining which chunks have already been indexed.


In [21]:

import json
from datetime import datetime, timezone
from langchain_core.documents import Document

BATCH_SIZE = 500

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def save_checkpoint(checkpoint, path):
    """Save checkpoint atomically."""
    temp_path = path.with_suffix(".tmp")

    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(checkpoint, f, indent=2)

    temp_path.replace(path)


def prepare_chroma_document(doc):
    """Convert metadata into Chroma-compatible values."""
    metadata = {}

    for key, value in doc.metadata.items():
        if value is None:
            continue

        if isinstance(value, (list, dict)):
            value = json.dumps(value, ensure_ascii=False)

        metadata[key] = value

    return Document(
        page_content=doc.page_content,
        metadata=metadata
    )


def resumable_indexing(documents, vectorstore, checkpoint_path, index_name):

    total = len(documents)
    chunk_ids = [doc.metadata["chunk_id"] for doc in documents]

    if len(chunk_ids) != len(set(chunk_ids)):
        raise ValueError(f"Duplicate chunk IDs in {index_name}")

    checkpoint = {
        "index_name": index_name,
        "total_documents": total,
        "batch_size": BATCH_SIZE,
        "completed_batches": [],
        "updated_at": None
    }

    if checkpoint_path.exists():
        with open(checkpoint_path, "r", encoding="utf-8") as f:
            checkpoint = json.load(f)

        print(f"Resuming {index_name} from existing checkpoint.")

    completed_batches = set(checkpoint.get("completed_batches", []))
    newly_added = 0

    for start in range(0, total, BATCH_SIZE):

        batch = documents[start:start + BATCH_SIZE]
        batch_ids = [doc.metadata["chunk_id"] for doc in batch]

        # Check actual Chroma records, not just checkpoint status
        existing_ids = set(
            vectorstore.get(ids=batch_ids)["ids"]
        )

        missing_docs = [
            doc for doc in batch
            if doc.metadata["chunk_id"] not in existing_ids
        ]

        # Insert only missing records
        if missing_docs:
            prepared_docs = [
                prepare_chroma_document(doc)
                for doc in missing_docs
            ]

            missing_ids = [
                doc.metadata["chunk_id"]
                for doc in missing_docs
            ]

            vectorstore.add_documents(
                documents=prepared_docs,
                ids=missing_ids
            )

            newly_added += len(missing_docs)

        # Verify the entire batch exists in Chroma
        verified_ids = set(
            vectorstore.get(ids=batch_ids)["ids"]
        )

        if len(verified_ids) != len(batch_ids):
            raise RuntimeError(
                f"Batch verification failed at position {start}"
            )

        completed_batches.add(start)

        # Save checkpoint after successful verification
        checkpoint.update({
            "index_name": index_name,
            "total_documents": total,
            "batch_size": BATCH_SIZE,
            "completed_batches": sorted(completed_batches),
            "newly_added_this_run": newly_added,
            "indexed_records": vectorstore._collection.count(),
            "updated_at": datetime.now(timezone.utc).isoformat()
        })

        save_checkpoint(checkpoint, checkpoint_path)

        print(
            f"{index_name}: "
            f"{min(start + len(batch), total)}/{total} "
            f"| Newly added: {len(missing_docs)}"
        )

    print(f"\n{index_name} indexing completed.")
    print("Collection count:", vectorstore._collection.count())
    print("New records added this run:", newly_added)


print("Checkpoint-based indexing configured.")


Checkpoint-based indexing configured.



## Step 25: Run Complete Resumable Vector Indexing Pipeline

**Purpose:**
Sequentially index discovery and detail datasets into their respective Chroma collections.

Execution order:
1. Discovery indexing
2. Detail indexing
3. Final collection verification

Both indexes use independent persistent checkpoints. Interrupted batches are reconciled against actual Chroma records before resuming.


In [22]:

from datetime import datetime

pipeline_start = datetime.now()

print("=" * 65)
print("SEVA_AI — COMPLETE VECTOR INDEXING PIPELINE")
print("=" * 65)

# 1. Discovery indexing
print("\n[1/2] STARTING DISCOVERY INDEXING")
print("-" * 65)

resumable_indexing(
    documents=discovery_chunks,
    vectorstore=discovery_db,
    checkpoint_path=CHECKPOINT_DIR / "discovery_indexing_checkpoint.json",
    index_name="seva_discovery_v1"
)

# 2. Detail indexing
print("\n[2/2] STARTING DETAIL INDEXING")
print("-" * 65)

resumable_indexing(
    documents=detail_chunks,
    vectorstore=detail_db,
    checkpoint_path=CHECKPOINT_DIR / "detail_indexing_checkpoint.json",
    index_name="seva_detail_v1"
)

# 3. Final verification
pipeline_end = datetime.now()

print("\n" + "=" * 65)
print("FINAL INDEXING REPORT")
print("=" * 65)

print("Discovery:", discovery_db._collection.count(), "/", len(discovery_chunks))
print("Detail:", detail_db._collection.count(), "/", len(detail_chunks))

print("Total indexed:",
      discovery_db._collection.count() + detail_db._collection.count())

print("Expected total:", len(discovery_chunks) + len(detail_chunks))

print("Started:", pipeline_start)
print("Completed:", pipeline_end)
print("Duration:", pipeline_end - pipeline_start)

print("\nPIPELINE COMPLETED SUCCESSFULLY")


SEVA_AI — COMPLETE VECTOR INDEXING PIPELINE

[1/2] STARTING DISCOVERY INDEXING
-----------------------------------------------------------------
seva_discovery_v1: 500/23438 | Newly added: 500
seva_discovery_v1: 1000/23438 | Newly added: 500
seva_discovery_v1: 1500/23438 | Newly added: 500
seva_discovery_v1: 2000/23438 | Newly added: 500
seva_discovery_v1: 2500/23438 | Newly added: 500
seva_discovery_v1: 3000/23438 | Newly added: 500
seva_discovery_v1: 3500/23438 | Newly added: 500
seva_discovery_v1: 4000/23438 | Newly added: 500
seva_discovery_v1: 4500/23438 | Newly added: 500
seva_discovery_v1: 5000/23438 | Newly added: 500
seva_discovery_v1: 5500/23438 | Newly added: 500
seva_discovery_v1: 6000/23438 | Newly added: 500
seva_discovery_v1: 6500/23438 | Newly added: 500
seva_discovery_v1: 7000/23438 | Newly added: 500
seva_discovery_v1: 7500/23438 | Newly added: 500
seva_discovery_v1: 8000/23438 | Newly added: 500
seva_discovery_v1: 8500/23438 | Newly added: 500
seva_discovery_v1: 9000